# Запуск пайплайна в Google Colab

**Каждый раз, когда открываете ноутбук или Colab перезапустился**, выполните ячейки 1–3 по порядку: подключение Диска, пути и свежая версия кода. Потом запускайте нужный шаг.

Colab удаляет свою среду, если ноутбук простаивает больше часа-полутора, а также не позже чем через 12 часов. Старые выводы ячеек при этом остаются на экране, поэтому кажется, что всё на месте. Если шаг пишет «Colab перезапустился», просто начните с ячейки 1.

Данные и результаты сохраняются на Google Диск и при перезапуске не теряются. Код каждый раз скачивается заново с GitHub — так всегда используется последняя версия.

Открывайте ноутбук по ссылке из репозитория, а не из сохранённой копии: в копии останутся старые ячейки.

## 1. Подключить Google Диск

Появится окно с запросом доступа — выберите аккаунт и разрешите. Дождитесь строки `Mounted at /content/drive`.

Если появится ошибка `Mountpoint must not already contain files`: меню «Среда выполнения» → «Отключиться и удалить среду выполнения», затем снова с ячейки 1. Ничего не удаляйте вручную: внутри `/content/drive` лежат файлы вашего Диска.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Пути к данным и результатам

In [ ]:
import os

REPO_URL = 'https://github.com/vardov337/crypto-news-forecasting.git'
DATA_DIR = '/content/drive/MyDrive/cryptonews-data'
RESULTS_DIR = '/content/drive/MyDrive/cryptonews-results'

assert os.path.ismount('/content/drive'), 'Диск не подключён — выполните ячейку 1'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ['CRYPTONEWS_DATA_DIR'] = DATA_DIR
os.environ['CRYPTONEWS_RESULTS_DIR'] = RESULTS_DIR
print('Данные:', DATA_DIR)
print('Результаты:', RESULTS_DIR)

## 3. Свежая версия кода

В конце должна напечататься версия с датой. Около минуты.

In [ ]:
%cd /content
!rm -rf /content/repo
!git clone -q $REPO_URL /content/repo
%cd /content/repo
!pip install -q -r requirements.txt
!pip install -q -e .
print('Версия:', open('VERSION').read().strip())

---
## Шаги пайплайна

Запускайте только тот шаг, о котором договорились. Повторный запуск любого шага безопасен: скачанное берётся с Диска, а не качается заново.

### Шаг 0. Проверка окружения
Несколько секунд. Фиксирует версии библиотек.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/00_check_env.py

### Шаг 1. Цены Binance
Первый раз 5–15 минут, повторно — 1–2 минуты (архивы уже на Диске).

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/01_download_prices.py

### Шаг 2. Новости ForkLog
Первый раз около часа. Повторный запуск продолжает с места остановки и уже собранное не скачивает — около минуты.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/02_collect_forklog.py

### Шаг 3. Новости CryptoVision
Перед первым запуском положите архив набора на Диск в папку `cryptonews-data/raw/cryptovision/`: Mendeley не даёт скачивать его автоматически. Архив скачивается кнопкой «Download All» на странице https://data.mendeley.com/datasets/3c3xtxtfb6/2, распаковывать его не нужно.

Шаг занимает несколько минут: распаковка, сверка часового пояса по 15-минутным свечам Binance, очистка.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/03_prepare_news_en.py

### Шаг 4. Очистка новостей ForkLog
Несколько секунд, сеть не нужна.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/04_prepare_news_ru.py

### Шаг 5. Тональность и файлы для ручной разметки
**Нужна видеокарта.** До ячеек 1–3: меню «Среда выполнения» → «Сменить среду выполнения» → «Графический процессор T4» → «Сохранить». После смены среды Colab перезапускается, поэтому ячейки 1–3 выполняются после неё.

Первый запуск — 10–20 минут: скачиваются шесть моделей и оцениваются все заголовки. Файлы для разметки создаются в самом начале шага, ещё до моделей. Предупреждение про `HF_TOKEN` можно не читать: модели открытые.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!nvidia-smi -L
!cd /content/repo && python scripts/05_sentiment.py

#### Скачать файлы для разметки
Скачивает на компьютер архив `annotation_files.zip`: четыре файла Excel и инструкция. Если браузер спросит, разрешите скачивание.

In [ ]:
import os
from google.colab import files
path = os.path.join(os.environ.get('CRYPTONEWS_RESULTS_DIR', ''), 'annotation', 'annotation_files.zip')
assert os.path.exists(path), 'Архива нет: выполните ячейки 1–3 и шаг 5'
files.download(path)

### Шаг 5b. Проверка моделей на ручной разметке
Запускать после того, как разметка загружена в репозиторий (папка `annotation/labels/`). Несколько секунд, видеокарта не нужна.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/05b_choose_sentiment.py

### Шаг 6. Признаки
Запускать после шага 5b. Около минуты, видеокарта не нужна. Строит таблицы признаков для биткоина и эфира и записывает даты окна настройки и тестовых фолдов.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/06_build_features.py

### Шаг 7. Модели
Нужна видеокарта (как для шага 5): «Среда выполнения» → «Сменить среду выполнения» → «Графический процессор T4», затем ячейки 1–3.

Сначала — пробный прогон: все модели на маленьком куске данных, около минуты. Он проверяет, что всё работает, его результаты в статью не идут.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/07_run_models.py --smoke

#### 7a. Базовые модели и ARIMA
Нулевой прогноз, историческое среднее, ARIMA и ARIMAX. Около часа, считается на процессоре.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/07_run_models.py --family baselines
!cd /content/repo && python scripts/07_run_models.py --family arima

#### 7b. Random Forest и XGBoost
На видеокарте около полутора часов. Если сессия оборвётся, выполните ячейки 1–3 и запустите эту ячейку снова: посчитанное не пересчитывается.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/07_run_models.py --family trees

#### 7c. LSTM
На видеокарте один-два часа. После обрыва — так же: ячейки 1–3 и снова эта ячейка.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/07_run_models.py --family lstm

### Шаг 8. Оценка моделей
Запускать, когда закончены 7a, 7b и 7c. Несколько минут, видеокарта не нужна. Считает точность прогнозов, статистические тесты, торговые стратегии с издержками и применяет критерий выбора модели.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/08_evaluate.py

### Шаг 8б. Проверки на утечку
Тест сдвига новостей и плацебо-тест. С видеокартой — несколько минут.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/08b_leakage_checks.py

### Шаг 8в. Тест Грейнджера
Проверка прогностической связи новостей и доходности в обе стороны, со стационарностью рядов и выбором числа лагов. Видеокарта не нужна, 1–3 минуты.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/08c_granger.py

### Шаг 9. Таблицы и рисунки для статьи
Запускать после шагов 8, 8б и 8в. Видеокарта не нужна, около минуты (в первый раз — несколько минут). Результат — папка `report` в результатах на Диске: `tables.xlsx`, рисунки статьи `fig1`–`fig6` и список версий пакетов `requirements-lock.txt`; всё вместе — архив `report.zip`.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/09_make_report.py

#### Скачать таблицы и рисунки
Скачивает на компьютер архив `report.zip`. Если браузер спросит, разрешите скачивание.

In [ ]:
import os
from google.colab import files
path = os.path.join(os.environ.get('CRYPTONEWS_RESULTS_DIR', ''), 'report.zip')
assert os.path.exists(path), 'Архива нет: выполните ячейки 1–3 и шаг 9'
files.download(path)

---
## Посмотреть результаты

Список таблиц для статьи, которые уже готовы.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
import glob
for path in sorted(glob.glob(f'{RESULTS_DIR}/**/*.csv', recursive=True)):
    print(path.replace(RESULTS_DIR + '/', ''))